# Fetch NMDC Pfam Annotation GFF — manifest builder (issue #88)

This notebook **only builds the URL manifest**. HTTP downloads run separately
via `scripts/download_to_cache.py` (multi-hour, would die if launched in-kernel).

Three-stage pipeline:

1. **This notebook** (Spark kernel, fast) → queries `nmdc_metadata.data_object_set`
   and writes `loaded_pfam_gff/manifest.csv`.
2. **`scripts/download_to_cache.py`** (terminal, hours) → reads `manifest.csv`,
   downloads each GFF with retries to `loaded_pfam_gff/raw_cache/`. Resumable.
3. **`parse_pfam_gff.ipynb`** (any kernel) → streaming parse → one Parquet for
   `nmdc_results.pfam_annotation_gff`. No network.

| `data_object_type` | ~files | ~total | ~avg | Format |
|---|---|---|---|---|
| `Pfam Annotation GFF` | 4,842 | 650 GB | 134 MB | 9-col TSV (no header) |

**Disk footprint:** ~650 GB of raw downloads will be cached under `OUT_DIR/raw_cache/`.
Override `DOWNLOAD_CACHE_DIR` to point at a scratch volume; delete the dir to
reclaim space after the Parquet write.

**Estimated output:** ~3 billion rows after parsing → `nmdc_results.pfam_annotation_gff`.

### Imports + config + logger + Spark

> **BERDL kernel note:** `get_spark_session()` is auto-imported by the BERDL JupyterHub kernel startup script. If you run this notebook outside BERDL, import it yourself.

In [ ]:
import logging
import os
from datetime import datetime
from pathlib import Path

OUT_DIR = Path(os.environ.get("PFAM_GFF_OUT_DIR", "loaded_pfam_gff"))
OUT_DIR.mkdir(exist_ok=True)

# Raw downloads will land here when scripts/download_to_cache.py runs.
# Override DOWNLOAD_CACHE_DIR to put them on a different volume.
CACHE_DIR = Path(os.environ.get("DOWNLOAD_CACHE_DIR", str(OUT_DIR / "raw_cache")))
CACHE_DIR.mkdir(parents=True, exist_ok=True)

LOG_DIR  = OUT_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)
LOG_FILE = LOG_DIR / f"fetch_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logger = logging.getLogger("nmdc_lakehouse.pfam_gff.fetch")
logger.setLevel(logging.INFO)
for _h in list(logger.handlers):
    logger.removeHandler(_h)
_fh = logging.FileHandler(LOG_FILE)
_fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
logger.addHandler(_fh)
_sh = logging.StreamHandler()
_sh.setFormatter(logging.Formatter("%(message)s"))
logger.addHandler(_sh)
logger.propagate = False

print(f"OUT_DIR:   {OUT_DIR.resolve()}")
print(f"CACHE_DIR: {CACHE_DIR.resolve()}")
print(f"LOG_FILE:  {LOG_FILE.resolve()}")

spark = get_spark_session(app_name="fetch_pfam_gff")
print(f"Spark version: {spark.version}")

### Fetch the URL manifest from `data_object_set`

In [ ]:
import time

import pandas as pd

TARGET_TYPE = "Pfam Annotation GFF"

manifest_sql = f"""
SELECT id, url, data_object_type, was_generated_by, file_size_bytes, md5_checksum
FROM nmdc_metadata.data_object_set
WHERE data_object_type = '{TARGET_TYPE}'
  AND url IS NOT NULL
  AND url LIKE 'https://data.microbiomedata.org/%'
ORDER BY id
"""

t0 = time.monotonic()
manifest = spark.sql(manifest_sql).toPandas()
logger.info(f"manifest fetched in {time.monotonic() - t0:.1f}s")

# Dedup: data_object_set sometimes has multiple ids pointing at the same URL.
n_before = len(manifest)
manifest = manifest.drop_duplicates(subset=["url"]).reset_index(drop=True)
if len(manifest) != n_before:
    logger.info(f"deduped: {n_before:,} → {len(manifest):,} (dropped {n_before - len(manifest):,} duplicates)")

# Drop zero-byte placeholder files (only ~7 of these in the current snapshot).
manifest["file_size_num"] = pd.to_numeric(manifest["file_size_bytes"], errors="coerce").fillna(0)
n_zero = int((manifest["file_size_num"] == 0).sum())
if n_zero:
    logger.info(f"dropping {n_zero:,} zero-byte placeholder files")
    manifest = manifest[manifest["file_size_num"] > 0].reset_index(drop=True)
manifest = manifest.drop(columns=["file_size_num"])

logger.info(f"{len(manifest):,} unique data objects to process")
size_gb = pd.to_numeric(manifest["file_size_bytes"], errors="coerce").fillna(0).sum() / 1024**3
logger.info(f"size estimate: {size_gb:.1f} GB total")

### Save the manifest

Writes `manifest.csv` next to this notebook's output dir. The next cell prints the exact terminal command for `scripts/download_to_cache.py`.

In [ ]:
manifest_csv = OUT_DIR / "manifest.csv"
manifest.to_csv(manifest_csv, index=False)
print(f"manifest written: {manifest_csv.resolve()}  ({len(manifest):,} rows)")

# Locate scripts/download_to_cache.py by walking up from cwd.
def _find_repo_root() -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "scripts" / "download_to_cache.py").exists():
            return parent
    raise FileNotFoundError(
        "scripts/download_to_cache.py not found in any parent of "
        f"{Path.cwd()}"
    )

repo_root  = _find_repo_root()
downloader = repo_root / "scripts" / "download_to_cache.py"
log_path   = (OUT_DIR / "download.log").resolve()

print()
print("To run the standalone downloader, paste this into a terminal:")
print()
print(f"  nohup python {downloader} \\")
print(f"      --manifest {manifest_csv.resolve()} \\")
print(f"      --cache-dir {CACHE_DIR.resolve()} \\")
print(f"      --workers 8 \\")
print(f"      > {log_path} 2>&1 &")
print()
print(f"  tail -f {log_path}")

## Next steps

The manifest is on disk. Run, **from a terminal**:

```bash
cd /home/mamillerpa/nmdc-lakehouse
nohup python scripts/download_to_cache.py \
    --manifest notebooks/loaded_pfam_gff/manifest.csv \
    --cache-dir notebooks/loaded_pfam_gff/raw_cache \
    --workers 8 \
    > notebooks/loaded_pfam_gff/download.log 2>&1 &

tail -f notebooks/loaded_pfam_gff/download.log
```

650 GB at typical pod bandwidth runs many hours — likely an overnight job.
When it finishes, open **`parse_pfam_gff.ipynb`** to parse the cache and write
the Parquet, then **`ingest_pfam_gff.ipynb`** to register the Delta table
under `nmdc_results`.